In [1]:
import pandas as pd
import numpy as np
import joblib

from sentence_transformers import SentenceTransformer

from sklearn.neighbors import NearestNeighbors

C:\Users\Shristy\anaconda3\envs\ai-support\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv(
    "../data/processed/issue_classification.csv"
)

df.head()

,text,label
0,I am logged in on my desktop but the mobile ap...,account_access
1,I would love to see a dark mode option for the...,feature_request
2,I was billed for an 'Onboarding Package' even ...,billing_problem
3,I am a developer and I want to know if there i...,billing_problem
4,I just upgraded my phone and I cannot figure o...,account_access


In [4]:
df.shape

(1316, 2)

In [5]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2127.01it/s]


In [6]:
embeddings = embedding_model.encode(

    df["text"].tolist(),

    show_progress_bar=True,

    convert_to_numpy=True
)

Batches: 100%|██████████| 42/42 [00:08<00:00,  5.17it/s]


In [7]:
nn = NearestNeighbors(

    n_neighbors=5,

    metric="cosine"

)

nn.fit(embeddings)

,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [8]:
np.save(

    "../models/ticket_embeddings.npy",

    embeddings

)

joblib.dump(

    nn,

    "../models/similarity_model.pkl"

)

df.to_csv(

    "../models/ticket_dataset.csv",

    index=False

)

with open(

    "../models/embedding_model_name.txt",

    "w"

) as f:

    f.write("all-MiniLM-L6-v2")

In [9]:
query = "I cannot login because my OTP never arrives."

In [10]:
query_embedding = embedding_model.encode(

    [query],

    convert_to_numpy=True
)

In [11]:
distances, indices = nn.kneighbors(
    query_embedding
)

In [12]:
threshold = 0.50

mask = (1 - distances[0]) >= threshold

results = df.iloc[indices[0][mask]].copy()

results["Similarity"] = (
    (1 - distances[0][mask]) * 100
)

results

,text,label,Similarity
949,Every time I try to verify my identity it says...,account_access,50.245762


In [13]:
print(distances)

[[0.49754238 0.5089766  0.51192176 0.51281786 0.52323616]]


In [14]:
print(embeddings.shape)

(1316, 384)


In [15]:
print(embedding_model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [16]:
query = df.iloc[100]["text"]

print(query)

Can you tell me more about the history of the company and who the founders are?


In [17]:
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

distances, indices = nn.kneighbors(query_embedding)

results = df.iloc[indices[0]].copy()
results["Similarity"] = (1 - distances[0]) * 100

results

,text,label,Similarity
100,Can you tell me more about the history of the ...,other,100.000000
296,How do I find the official social media accoun...,other,38.540607
746,How do I find the contact information for your...,other,37.525028
818,I’m a student and I’m looking for some case st...,other,36.347145
650,I noticed that the company logo in the top lef...,bug,31.108307
